In [ ]:
# Import
import random
import numpy as np
import torch
import json
from tqdm import tqdm
from pathlib import Path
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
import os
import csv
from transformers import RobertaModel, RobertaTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from collections import Counter, defaultdict, deque
import re
from rank_bm25 import BM25Okapi
import json
import csv

device = "cuda" if torch.cuda.is_available() else "cpu"

# Seed for reproductibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Default paths
ROOT = Path("../Amazon_products") # Root Amazon_products directory
CLASS_HIERARCHY_PATH = ROOT / "class_hierarchy.txt" 

def load_multilabel(path):
    """Load multi-label data into {id: [labels]} dictionary -> for class_hierarchy"""
    id2labels = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                pid, label = parts
                pid = int(pid)
                label = int(label)
                if pid not in id2labels:
                    id2labels[pid] = []
                id2labels[pid].append(label)
    return id2labels

class2hierarchy = load_multilabel(CLASS_HIERARCHY_PATH) # id parents -> children (taxonomy)
print(list(class2hierarchy.items())[:1])
print(len(class2hierarchy)) #69


c:\Users\noamc\Documents\insa_korea\Cours\big data\final proj\project_release\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[(0, [1, 8, 208, 211, 213, 216, 229, 255, 265, 218, 271, 277, 249, 288, 313, 357])]
69


In [2]:
def label_coverage(silver_labels, num_classes=531):
    """
    silver_labels : { review_id: [label1, label2, ...] }
    returns coverage_ratio, covered_classes
    """
    covered = set()

    for i, labels in silver_labels.items():
        for lbl in labels:
            if 0 <= lbl < num_classes:
                covered.add(lbl)

    coverage_ratio = len(covered) / num_classes
    return coverage_ratio, sorted(list(covered))


def load_silver_files(paths: dict):
    """Load multiple silver files: name -> {pid -> {labels, scores}}"""
    silvers = {}
    for name, path in paths.items():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        silvers[name] = {
            int(pid): {"labels": d["labels"],"scores": d["scores"]}
            for pid, d in data.items()
        }

        print(f"Loaded {name}: {len(silvers[name])} samples")

    return silvers

def load_silver_files_csv(paths: dict, offset=29487):
    """
    Loads multiple CSV files and applies an offset to the PID for test.
    name -> { pid+offset -> {labels, scores} }
    """
    silvers = {}

    for name, path in paths.items():
        d = {}

        with open(path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                pid = int(row["id"]) + offset 
                lbl_str = row["label"].strip().replace('"', "")
                if lbl_str == "":
                    labels = []
                else:
                    labels = [int(x) for x in lbl_str.split(",")]
                fixed_scores = [1.0, 0.7, 0.5]
                scores = [fixed_scores[i] if i < len(fixed_scores) else 0.5 for i in range(len(labels))]
                d[pid] = {"labels": labels, "scores": scores}

        silvers[name] = d

    return silvers


def hierarchy_consistency(silver, hierarchy):
    """Hierarchy consistency in a hierarchy given for our silver labels"""
    ok = 0
    total = 0
    for labels in silver.values():
        L = set(labels)
        for parent, children in hierarchy.items():
            for child in children:
                if child in L:
                    total += 1
                    if parent in L:
                        ok += 1
    return ok / total if total > 0 else 0


def majority_vote(silvers: dict, top_k=5, min_labels=2, total=False):
    """
    Majority vote but keeps the BEST SCORE per label across models.
    silvers[name][pid] = {"labels": [...], "scores": [...]}
    """
    model_count = len(silvers)
    if total == False: 
        majority = model_count // 2 + 1
    else:
        majority = model_count
    print(f"\nUsing strict majority: need {majority}/{model_count} votes\n")

    # PIDs shared by all models
    all_pids = set.intersection(
        *(set(d.keys()) for d in silvers.values())
    )

    clean = {}

    for pid in all_pids:
        vote_counts = {}
        best_scores = {}

        # Count votes + track best score
        for model_name, data in silvers.items():
            labels = data[pid]["labels"]
            scores = data[pid]["scores"]

            for lab, sc in zip(labels, scores):
                vote_counts[lab] = vote_counts.get(lab, 0) + 1
                best_scores[lab] = max(best_scores.get(lab, -1), sc)

        # Keep labels with majority
        kept_labels = sorted([lab for lab, cnt in vote_counts.items() if cnt >= majority])

        if len(kept_labels) < min_labels:
            continue

        # Cut to top_k (based on best score)
        kept_labels = sorted(kept_labels,key=lambda x: best_scores[x],reverse=True)[:top_k]

        kept_scores = [ float(best_scores[lab]) for lab in kept_labels ]

        clean[pid] = {"labels": kept_labels, "scores": kept_scores}

    print(f"Cleaned samples: {len(clean)}")
    return clean

In [ ]:
# create here other combinations if needed

paths6 = { # all remake
    "roberta": "SilverRemake/silver_train_roberta.json",
    "mpnet": "SilverRemake/silver_train_mpnet.json",
    "mini": "SilverRemake/silver_train_mini.json",
    "BM": "SilverRemake/silver_train_mixBM.json",
}

# care csv or json !!!
silvers = load_silver_files(paths6)
#silvers = load_silver_files_csv(paths4)

# Majority vote
# total=True -> all models must agree
# total=False -> majority vote
clean = majority_vote(silvers, top_k=3, min_labels=2, total=False)

OUT_PATH = "SilverCombo/pseudo_gold_all.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(clean, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {OUT_PATH}")


silver_train_labels = {pid: info["labels"] for pid, info in clean.items()}
consistency = hierarchy_consistency(silver_train_labels, class2hierarchy)
print(f"\nHierarchy Consistency: {consistency:.2%}")

coverage, classes = label_coverage(silver_train_labels)
print(f"Coverage: {coverage:.2%}")
print(f"Covered classes: {len(classes)}/531")

# check gold
ids = [29487, 29488, 29489, 29490, 29491]
for pid in ids:
    if pid in silver_train_labels:
        print(f"{pid} -> {silver_train_labels[pid]}")
    else:
        continue

Loaded roberta: 49145 samples
Loaded mpnet: 49145 samples
Loaded mini: 49145 samples
Loaded BM: 49145 samples

Using strict majority: need 4/4 votes

Cleaned samples: 13479

Saved: SilverCombo/unique_gold_all.json

Hierarchy Consistency: 0.00%
Coverage: 80.23%
Covered classes: 426/531
29487 -> [93]
29488 -> [17]
